# Etapa 05 — Baseline temporal

Execução real informada pelo autor em 2026-10-07, com outputs preservados abaixo.
Run `f6d7aca720f74316b4183f97b6d866ab`; commit e CI do baseline ainda não informados.
[Contrato e resultados](../../docs/BASELINE.md), [protocolo](../../docs/EVALUATION_PROTOCOL.md),
[preparação](../../references/evidence/baseline_preparation_2026-10-07.json) e
[execução do autor](../../references/evidence/baseline_validation_2026-10-07.json)
registram evidências de origens distintas.


## Hipótese e candidatos

A Gold real foi construída pelo autor: 268.668 linhas de treino, 67.255 de validação
 e 66.954 de teste, com 19 preditores. Evidência:
[recibo Gold](../../references/evidence/gold_build_2026-10-07.json).

O controle `dummy_prior` não usa sinal. A regressão logística usa escala e L2;
o gradient boosting capta relações não lineares. Os dois usam pesos de classe
calculados no treino. Configurações e seed 42 são fixas, sem busca ou early stopping
aleatório. Escala fica dentro do pipeline; não há remoção de zeros/extremos.


## Fronteira da avaliação

Treino ajusta modelos e scaler. Validação compara AP, ROC AUC e precisão diária
por cliente. O risco diário do cliente é o máximo score; qualquer fraude no dia
marca o cliente como positivo. Desempate por ID crescente e limite de cem.

Somente AP seleciona entre os dois candidatos, com desempate pelo nome do modelo.
O dummy é controle. O relatório verifica se o candidato supera esse controle.
O teste permanece reservado: integridade estrutural é conferida, mas não há
fit, scores ou métricas de teste. O vencedor permanece ajustado no treino, sem refit.
Scores não são probabilidades calibradas, e esta entrega não escolhe um limiar.


## Execute no terminal e selecione explicitamente uma execução

```bash
poetry install
make validate
poetry run python -m fraud_detection_mlops.modeling.train run
poetry run python -m fraud_detection_mlops.modeling.train verify "<baseline_path>"
```

Use o `run_id` retornado na célula abaixo. Não buscamos automaticamente a última
pasta: uma comparação precisa apontar para uma execução específica.
As células só leem outputs existentes; os resultados abaixo foram executados
localmente pelo autor. Artefatos nativos ficam sob `data/`, fora do Git.


In [2]:
import json
import re

from fraud_detection_mlops.bronze import DEFAULT_INVENTORY, load_inventory
from fraud_detection_mlops.modeling.train import DEFAULT_BASELINE, verify_baseline

RUN_ID = "f6d7aca720f74316b4183f97b6d866ab"
if not re.fullmatch(r"[0-9a-f]{32}", RUN_ID):
    raise ValueError("Defina RUN_ID com o identificador da execução local")

source_commit = load_inventory(DEFAULT_INVENTORY)["source"]["commit"]
run_dir = DEFAULT_BASELINE / source_commit / "baseline_v1" / RUN_ID
print(verify_baseline(run_dir))
report = json.loads((run_dir / "report.json").read_text())

{'baseline_path': '/home/wanderson/Documentos/fraud-detection-mlops/data/processed/handbook/6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a/baseline_v1/f6d7aca720f74316b4183f97b6d866ab', 'run_id': 'f6d7aca720f74316b4183f97b6d866ab', 'verified_outputs': 10, 'selected_model': 'hist_gradient_boosting', 'comparison': {'dummy_prior': {'rows': 67255, 'fraud_count': 580, 'fraud_rate': 0.008623894134265111, 'average_precision': 0.008623894134265111, 'roc_auc': 0.5, 'daily_customer_precision_at_100': 0.02142857142857143}, 'logistic_regression': {'rows': 67255, 'fraud_count': 580, 'fraud_rate': 0.008623894134265111, 'average_precision': 0.4350008973626218, 'roc_auc': 0.8731684315322654, 'daily_customer_precision_at_100': 0.48000000000000004}, 'hist_gradient_boosting': {'rows': 67255, 'fraud_count': 580, 'fraud_rate': 0.008623894134265111, 'average_precision': 0.6239485132514692, 'roc_auc': 0.8849532601528257, 'daily_customer_precision_at_100': 0.5399999999999999}}, 'test_evaluated': False, 'status': 's

In [3]:
import pandas as pd

comparison = pd.DataFrame.from_dict(report["comparison"], orient="index")
columns = ["average_precision", "roc_auc", "daily_customer_precision_at_100", "fraud_rate", "rows"]
print(comparison[columns].to_string())
print("Selecionado na validação:", report["selected_model"])
print("Superou AP do dummy:", report["selected_beats_dummy_ap"])
print("Teste avaliado:", report["test_evaluated"])

                        average_precision   roc_auc  daily_customer_precision_at_100  fraud_rate   rows
dummy_prior                      0.008624  0.500000                         0.021429    0.008624  67255
hist_gradient_boosting           0.623949  0.884953                         0.540000    0.008624  67255
logistic_regression              0.435001  0.873168                         0.480000    0.008624  67255
Selecionado na validação: hist_gradient_boosting
Superou AP do dummy: True
Teste avaliado: False


In [4]:
selected = report["selected_model"]
metrics = json.loads((run_dir / "models" / selected / "metrics.json").read_text())
print(pd.DataFrame(metrics["metrics"]["daily_customer_metrics"]).to_string(index=False))
print("Fit/predict (s):", metrics["fit_seconds"], metrics["predict_seconds"])

 alerts  customers       date  fraudulent_customers_in_alerts  precision_at_100
    100       3733 2018-05-06                              45              0.45
    100       3735 2018-05-07                              55              0.55
    100       3706 2018-05-08                              53              0.53
    100       3723 2018-05-09                              60              0.60
    100       3693 2018-05-10                              54              0.54
    100       3730 2018-05-11                              51              0.51
    100       3733 2018-05-12                              60              0.60
Fit/predict (s): 0.6027634560014121 0.036915013999532675


## Como interpretar a execução real

Compare AP com a prevalência e o controle constante. Considere também concentração
por cliente/dia, desempenho entre dias e o suporte da janela. Melhor AP na validação
é evidência de seleção entre estes candidatos, sem garantia de estabilidade futura.
Uma única semana não fundamenta significância estatística ou desempenho em produção.

Não examine o teste para decidir features, candidatos ou limiar. Primeiro registre
as métricas reais, investigue a comparação e congele as escolhas. Calibração,
ablação, busca temporal e finalização do teste terão contratos próprios.

## Proveniência e limites

Um joblib por modelo preserva seu pipeline; manifestos ligam Gold, código, contratos,
lockfile, métricas e scores. O verify recalcula métricas sem carregar objetos joblib.
Os outputs publicados das etapas anteriores são preservados. O assistente não
construiu a Gold real nem treinou estes candidatos no histórico completo do autor.

**Resultados reais:** AP 0,623949 do gradient boosting e precisão diária média
de 54%, contra AP 0,435001 e 48% da regressão. O teste não foi avaliado. A janela
de sete dias não confirma superioridade estatística. Checks locais: 127 testes;
commit e CI ainda não informados. A [etapa MLflow](../../docs/MLFLOW.md) exporta
esses modelos sem alterar a execução histórica. Não há evidência de eficácia
operacional em produção.
